# 03 -- Single-objective shape optimisation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antnewman/acoustic-beacon-optimiser/blob/main/notebooks/03_shape_optimisation.ipynb)

Maximise integrated conspicuousness (IC) over the spherical-cap family
subject to a surface-area constraint, using CMA-ES.

**What this shows:** given a bat call spectrum, the CMA-ES evolution
strategy finds the cap geometry (radius, half-angle) that maximises
conspicuousness without exceeding a given surface area. The optimum
can then be compared against the natural *Marcgravia* and *Mucuna*
geometries from the literature.

**Runtime:** around 10-30 minutes on a single CPU for a 100-200
evaluation budget at the default evaluation grid. For production
results, scale up `MAX_EVALS` and refine `frequencies` / `angles`.

**Setup in Colab:** run `!pip install -q git+https://github.com/antnewman/acoustic-beacon-optimiser.git` in the first cell, then continue.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from abo.biology.call_spectra import glossophaga_call_spectrum
from abo.biology.natural_shapes import MARCGRAVIA_EVENIA, MUCUNA_HOLTONII
from abo.optimisation.encoding import SphericalCapEncoding
from abo.optimisation.runners import EvaluationConfig, make_ic_cost
from abo.optimisation.single_objective import optimise_cmaes

## Problem setup

In [ ]:
AREA_MAX = 0.008  # 8000 mm^2 -- a generous upper bound for a bat-attracting reflector
MAX_EVALS = 150
SEED = 42

# Evaluation grid: coarse enough for reasonable runtime, fine enough for physics
frequencies = np.linspace(45_000.0, 100_000.0, 8)
angles = np.linspace(0.0, np.pi / 3, 7)
spectrum = glossophaga_call_spectrum(frequencies)

config = EvaluationConfig(
    frequencies=frequencies,
    angles=angles,
    call_spectrum=spectrum,
)

## Run CMA-ES over the spherical cap family

In [ ]:
encoder = SphericalCapEncoding()
bounds = encoder.default_bounds
sigma0 = 0.25 * np.mean([hi - lo for lo, hi in bounds])

cost = make_ic_cost(encoder, config, area_max=AREA_MAX)

result = optimise_cmaes(
    objective=cost,
    x0=encoder.x0,
    sigma0=sigma0,
    bounds=bounds,
    max_evaluations=MAX_EVALS,
    seed=SEED,
)

best_cap = encoder.decode(result.x_best)
print(f"Best cap:")
print(f"  radius      = {best_cap.radius*1000:.2f} mm")
print(f"  half_angle  = {np.degrees(best_cap.half_angle):.2f} deg")
print(f"  depth       = {best_cap.depth*1000:.2f} mm")
print(f"  aperture    = {best_cap.aperture*1000:.2f} mm")
print(f"  surface_area= {best_cap.surface_area*1e6:.1f} mm^2")
print(f"Best IC       = {-result.f_best:.2f} dB")
print(f"Evaluations   = {result.n_evals}")

## Convergence history

In [ ]:
history = np.array(result.history)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(history) + 1), -history, linewidth=1.5)
ax.set_xlabel("CMA-ES generation")
ax.set_ylabel("Best-so-far IC (dB)")
ax.set_title("Single-objective spherical cap optimisation")
ax.grid(visible=True, linestyle=":", linewidth=0.5)
fig.tight_layout()
fig.savefig("../paper/figures/fig03_cmaes_convergence.png", dpi=200)
plt.show()

## Compare against natural reflectors

Place the optimum on a `(radius, half-angle)` map alongside
the Marcgravia and Mucuna approximations. Natural reflectors
in the literature are reported as `(diameter, depth)`, which we
convert to the cap's `(sphere_radius, half_angle)` parameters.

In [ ]:
def natural_to_sphere(diameter_mm, depth_mm):
    """Convert (diameter, depth) to (sphere_radius, half_angle)."""
    a = diameter_mm / 2.0 * 1e-3
    d = depth_mm * 1e-3
    r = (a**2 + d**2) / (2.0 * d)
    theta = float(np.arcsin(a / r))
    return r, theta

natural = [
    (MARCGRAVIA_EVENIA, "Marcgravia evenia"),
    (MUCUNA_HOLTONII, "Mucuna holtonii"),
]

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    [best_cap.radius * 1000],
    [np.degrees(best_cap.half_angle)],
    marker="*", s=200, label="CMA-ES optimum",
)
for nat, name in natural:
    r, theta = natural_to_sphere(nat.diameter_mm, nat.depth_mm)
    ax.scatter([r * 1000], [np.degrees(theta)], label=name, s=80)
ax.set_xlabel("Sphere radius (mm)")
ax.set_ylabel("Half-angle (deg)")
ax.set_title("Optimum vs natural reflectors (spherical cap family)")
ax.grid(visible=True, linestyle=":", linewidth=0.5)
ax.legend()
fig.tight_layout()
fig.savefig("../paper/figures/fig03_optimum_vs_natural.png", dpi=200)
plt.show()